<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/5%EC%A3%BC%EC%B0%A8%20%EC%A0%95%EB%A6%AC%20%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# [5주차 복습과제] 파이썬 텍스트 분석 - Word2Vec과 맥락의 이해

## 1. Word2Vec에 대해서 이해하기
Word2Vec은 텍스트(단어)를 컴퓨터가 계산할 수 있는 공간상의 벡터(숫자 배열)로 변환하는 기술입니다.
기존 TF-IDF는 단어의 형태가 다르면 완전히 다른 단어로 취급했지만, Word2Vec은 단어
벡터 간의 **'코사인 유사도'**를 계산하여 "배우"와 "연기"처럼 의미가 비슷한 단어를 수치적으로 파악할 수 있게 해줍니다.

SyntaxError: invalid syntax (4179470036.py, line 2)

In [4]:
import numpy as np
from gensim.models import Word2Vec

# 실습을 위한 가상의 영화 리뷰 데이터 (형태소 분석 완료 가정)
sentences = [
    ["영화", "배우", "연기", "정말", "훌륭하다", "몰입", "최고"],
    ["주인공", "배우", "연기", "감동", "눈물", "나오다"],
    ["스토리", "지루하다", "배우", "연기", "아쉽다"],
    ["감독", "연출", "배우", "연기", "모두", "완벽하다"]
]

print("데이터 로드 완료! 총 문장 수:", len(sentences))

ModuleNotFoundError: No module named 'gensim'

In [ ]:
## 2. 맥락(Context)을 이해한다는 것에 대해서 파악하기
Word2Vec은 **"비슷한 문맥에서 함께 등장하는 단어는 비슷한 의미를 가진다"**는 가설을 바탕으로 학습합니다.
위 데이터에서 '연기'와 '배우'는 주변에 '훌륭하다', '몰입', '감동' 등의 단어와 함께 등장합니다.
모델은 이 **주변 단어들(맥락)**을 보고 두 단어가 유사하다고 판단합니다.

파이썬 `gensim` 라이브러리에서는 **`window`** 파라미터를 통해
"기준 단어의 앞뒤로 몇 개의 단어까지를 맥락으로 볼 것인지" 그 범위를 설정할 수 있습니다.

In [ ]:
# Word2Vec 모델 학습
# vector_size: 단어 벡터의 차원 (실전에서는 100~300)
# window: 문맥(Context)의 범위 (앞뒤 3단어까지 맥락으로 파악)
# min_count: 최소 등장 횟수
model = Word2Vec(sentences=sentences, vector_size=10, window=3, min_count=1, seed=42)

# 1. '맥락' 학습 결과 확인: 단어 간 유사도 도출
similarity_score = model.wv.similarity("배우", "연기")
print(f"▶ '배우'와 '연기'의 코사인 유사도: {similarity_score:.4f} (1.0에 가까울수록 문맥이 유사함)")

# 2. 특정 단어와 가장 비슷한 맥락을 가진 단어 찾기
similar_words = model.wv.most_similar("연기", topn=3)
print(f"▶ '연기'와 가장 맥락이 유사한 단어들: {similar_words}")

In [ ]:
## 3. 4주차 모델(분류기)과의 연결 (문서 벡터 생성)
Word2Vec은 단어 단위의 벡터를 생성하므로, 리뷰 전체(문서)의 감성을 분류하기 위해서는 문장에 포함된 **단어 벡터들의 평균**을 구하여 하나의
'문서 벡터'로 만들어 머신러닝 모델(로지스틱 회귀 등)에 입력해야 합니다.

In [ ]:
# 첫 번째 문장(리뷰)을 하나의 문서 벡터로 변환 (단어 벡터의 평균)
review_tokens = sentences[0]
doc_vector = np.mean([model.wv[word] for word in review_tokens], axis=0)

print("▶ 첫 번째 리뷰 텍스트:", review_tokens)
print("▶ 문서 벡터 변환 결과 (배열 크기):", doc_vector.shape)
print("▶ 문서 벡터 값 (일부):", np.round(doc_vector, 3))